<a href="https://colab.research.google.com/github/DrLuisZapata/lidar-cnn-lstm-curves/blob/main/PandaSet_Data_Exploration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PandaSet Data Exploration and LiDAR Feature Visualization

This notebook accompanies the article *"LiDAR-Based Speed and Steering Prediction on Curved Roads Using a Physics-Informed CNN-LSTM Network"*.

It covers the data-preparation and exploration steps on **PandaSet**:

1. Loading and visualizing a LiDAR point cloud.
2. Computing vehicle speed from the ego poses.
3. Ground-point filtering and a bird's-eye view (BEV) of the road boundaries.

Model training (CNN-LSTM with physics-informed loss) and evaluation on PandaSet and KITTI are in the companion notebook `KITTI_Validation_CNN_LSTM_PIDL.ipynb`.

**Data:** PandaSet sequences are expected in Google Drive under `/content/drive/MyDrive/pandaset/<sequence>/` (e.g., `001/lidar/00.pkl`, `001/camera/front_camera/poses.json`).

## 1. Setup

In [ ]:
# Install Open3D, used to build and color the LiDAR point cloud.
!pip install open3d

### Kaggle credentials (optional)

Run the next cell only if you download PandaSet through the Kaggle API. Add `KAGGLE_USERNAME` and `KAGGLE_KEY` in the Colab **Secrets** panel (key icon) first; see the repository README.

In [ ]:
# Configure Kaggle API credentials from Colab Secrets (or prompt for them if not set).
# The credentials are written to ~/.kaggle/kaggle.json, outside the notebook.

import json
import os
from getpass import getpass

try:
    from google.colab import userdata
    username = userdata.get("KAGGLE_USERNAME")
    key = userdata.get("KAGGLE_KEY")
except Exception:
    username = input("Kaggle username: ")
    key = getpass("Kaggle API key: ")

# Create the configuration file in the folder used by the Kaggle API, readable only by the owner
os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
config_path = os.path.expanduser("~/.kaggle/kaggle.json")
with open(config_path, "w") as f:
    json.dump({"username": username, "key": key}, f)
os.chmod(config_path, 0o600)

print("Kaggle credentials configured.")

In [ ]:
# Mount Google Drive and check that the PandaSet folder is available.

import os
from google.colab import drive

drive.mount('/content/drive')

# Root folder of the PandaSet sequences (change if your data are stored elsewhere)
PANDASET_ROOT = "/content/drive/MyDrive/pandaset/"

sequences = sorted(d for d in os.listdir(PANDASET_ROOT) if os.path.isdir(os.path.join(PANDASET_ROOT, d)))
print(f"Found {len(sequences)} sequences: {sequences[:10]}{' ...' if len(sequences) > 10 else ''}")

## 2. Coordinate-frame check

The analysis below interprets the LiDAR `x` axis as the forward distance and `y` as the lateral distance from the vehicle. This holds only if the points are expressed in the vehicle (ego) frame. This cell compares the point-cloud center with the LiDAR sensor pose in the first and last frames of a sequence: if the point-cloud center moves together with the sensor position, the points are stored in **world coordinates** and must be transformed to the ego frame (e.g., with the official PandaSet devkit) before the features are computed.

With PandaSet the center follows the sensor, so the points are in world coordinates; the next cells apply the transform.

In [ ]:
# Compare the point-cloud center with the LiDAR sensor position in the first and last frames.

import json
import numpy as np
import pandas as pd

SEQUENCE = "001"
lidar_dir = os.path.join(PANDASET_ROOT, SEQUENCE, "lidar")
lidar_files = sorted(f for f in os.listdir(lidar_dir) if f.endswith(".pkl"))

# LiDAR sensor poses (one per frame) provided by PandaSet
with open(os.path.join(lidar_dir, "poses.json"), "r") as f:
    lidar_poses = json.load(f)

for idx in [0, len(lidar_files) - 1]:
    pts = pd.read_pickle(os.path.join(lidar_dir, lidar_files[idx]))[["x", "y", "z"]].values
    center = np.median(pts[:, :2], axis=0)
    sensor = np.array([lidar_poses[idx]["position"]["x"], lidar_poses[idx]["position"]["y"]])
    print(f"Frame {idx:02d}: point-cloud center (x, y) = {center.round(1)} | "
          f"sensor position (x, y) = {sensor.round(1)}")

print("\nIf the point-cloud center follows the sensor position between frames, the points are in "
      "world coordinates and should be transformed to the ego frame before feature extraction.")

### World-to-vehicle transform

The check above shows that the point-cloud center moves together with the sensor, so PandaSet points are in **world coordinates**. The following functions transform each frame to a vehicle-aligned frame (origin at the sensor, `x` along the direction of travel, `y` to the left, `z` vertical). The same functions are used in the CNN-LSTM notebook.

In [ ]:
# World-to-vehicle transform (same implementation as in the CNN-LSTM notebook).

MIN_DESPLAZAMIENTO_M = 0.05   # below this displacement per frame (0.5 m/s) the yaw is held constant

def calcular_yaw_trayectoria(x, y, min_desp=MIN_DESPLAZAMIENTO_M):
    """
    Estimate the vehicle yaw (rad) at each frame from the sensor trajectory.

    The yaw at frame k is the direction of the displacement from frame k-1 to frame k
    (backward difference, so no future frame is used). When the displacement is too small
    (vehicle practically stopped), atan2 is meaningless and the previous yaw is kept.
    The result is unwrapped to avoid +/-pi jumps.
    """
    n = len(x)
    yaw = np.full(n, np.nan)
    ultimo = None
    for k in range(1, n):
        dx, dy = x[k] - x[k - 1], y[k] - y[k - 1]
        if np.hypot(dx, dy) >= min_desp:
            ultimo = np.arctan2(dy, dx)
        if ultimo is not None:
            yaw[k] = ultimo
    validos = ~np.isnan(yaw)
    if not validos.any():
        return np.zeros(n)
    # Frames before the first valid motion (including frame 0) take the first valid yaw
    yaw[~validos] = yaw[validos][0]
    return np.unwrap(yaw)


def mundo_a_vehiculo(puntos, origen_xy, yaw):
    """
    Transform world points to a vehicle-aligned frame: translate to the sensor position (x, y)
    and rotate by -yaw so that +x points along the direction of travel and +y to the left.
    The z coordinate (world vertical) is kept unchanged.
    """
    c, s = np.cos(yaw), np.sin(yaw)
    dx = puntos[:, 0] - origen_xy[0]
    dy = puntos[:, 1] - origen_xy[1]
    x_v = c * dx + s * dy
    y_v = -s * dx + c * dy
    return np.column_stack([x_v, y_v, puntos[:, 2]])



# Sensor trajectory and yaw for the sequence used in the figures
with open(os.path.join(lidar_dir, "poses.json"), "r") as f:
    _poses = json.load(f)
sensor_x = np.array([p["position"]["x"] for p in _poses])
sensor_y = np.array([p["position"]["y"] for p in _poses])
sensor_yaw = calcular_yaw_trayectoria(sensor_x, sensor_y)
print(f"Yaw at frame 0: {np.degrees(sensor_yaw[0]):.1f} deg (world frame)")

## 3. LiDAR point-cloud visualization

Loads one LiDAR frame, colors the points by height and displays an interactive 3D view (downsampled for browser performance).

In [ ]:
# Load one LiDAR frame and display it as an interactive 3D point cloud colored by height.

import open3d as o3d
import matplotlib.pyplot as plt
import plotly.graph_objects as go

# Folder for the article figures (PNG at 300 dpi)
FIG_DIR = "figures"
os.makedirs(FIG_DIR, exist_ok=True)

lidar_path = os.path.join(PANDASET_ROOT, SEQUENCE, "lidar", "00.pkl")

try:
    # PandaSet stores each LiDAR frame as a pickled DataFrame; keep the X, Y, Z columns
    lidar_df = pd.read_pickle(lidar_path)
    points = lidar_df[["x", "y", "z"]].values
    # Transform from world coordinates to the vehicle-aligned frame of frame 0
    points = mundo_a_vehiculo(points, (sensor_x[0], sensor_y[0]), sensor_yaw[0])

    # Build an Open3D point cloud
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(points)

    # Color each point by its normalized height (Z) using the viridis colormap
    z_values = points[:, 2]
    z_norm = (z_values - z_values.min()) / (z_values.max() - z_values.min())
    colors = plt.get_cmap("viridis")(z_norm)[:, :3]
    pcd.colors = o3d.utility.Vector3dVector(colors)

    print(f"Loaded {len(points)} points.")

    # Downsample (1 of every 5 points) so the interactive plot stays responsive
    step = 5
    points_ds = points[::step]
    colors_ds = colors[::step]

    fig = go.Figure(data=[go.Scatter3d(
        x=points_ds[:, 0], y=points_ds[:, 1], z=points_ds[:, 2],
        mode="markers",
        marker=dict(size=1, color=colors_ds)
    )])
    fig.update_layout(title=f"PandaSet LiDAR point cloud - sequence {SEQUENCE}, frame 00")
    fig.show()

    # Static version for the article (Plotly screenshots are low resolution):
    # 3D scatter with Matplotlib, colored by height, saved at 300 dpi.
    fig_s = plt.figure(figsize=(8, 6))
    ax_s = fig_s.add_subplot(111, projection="3d")
    sc = ax_s.scatter(points_ds[:, 0], points_ds[:, 1], points_ds[:, 2],
                      c=points_ds[:, 2], cmap="viridis", s=0.3)
    ax_s.set_xlabel("Forward x (m)")
    ax_s.set_ylabel("Lateral y (m)")
    ax_s.set_zlabel("z (m)")
    ax_s.set_title(f"PandaSet LiDAR point cloud - sequence {SEQUENCE}, frame 00")
    fig_s.colorbar(sc, ax=ax_s, shrink=0.6, label="Height z (m)")
    fig_s.savefig(os.path.join(FIG_DIR, "fig2_point_cloud.png"), dpi=300, bbox_inches="tight")
    plt.show()

except FileNotFoundError:
    print(f"File not found: {lidar_path}. Check the PandaSet path.")

## 4. Vehicle speed from ego poses

Computes the vehicle speed from the 3D displacement between consecutive poses (PandaSet is sampled at 10 Hz, i.e., 0.1 s between frames).

In [ ]:
# Compute vehicle speed (km/h) for one sequence from the front-camera poses.

poses_path = os.path.join(PANDASET_ROOT, SEQUENCE, "camera", "front_camera", "poses.json")

with open(poses_path, "r") as f:
    poses_data = json.load(f)

# Extract the X, Y, Z position of each frame
df_poses = pd.DataFrame(poses_data)
df_poses["x"] = df_poses["position"].apply(lambda p: p["x"])
df_poses["y"] = df_poses["position"].apply(lambda p: p["y"])
df_poses["z"] = df_poses["position"].apply(lambda p: p["z"])

# 3D distance between consecutive frames (Z included so that grades do not bias the speed)
distances = np.sqrt(
    df_poses["x"].diff() ** 2 +
    df_poses["y"].diff() ** 2 +
    df_poses["z"].diff() ** 2
)

# Speed = distance / time, with dt = 0.1 s (10 Hz), converted from m/s to km/h
DT = 0.1
df_poses["speed_kmh"] = (distances / DT) * 3.6

# The first frame has no previous position; drop it from the statistics
speeds = df_poses["speed_kmh"].dropna()

print(f"--- Speed statistics (sequence {SEQUENCE}) ---")
print(f"Mean speed: {speeds.mean():.2f} km/h")
print(f"Max speed:  {speeds.max():.2f} km/h")
print(f"Frames:     {len(df_poses)}")

# Speed profile over the sequence
plt.figure(figsize=(10, 4))
plt.plot(df_poses.index, df_poses["speed_kmh"])
plt.xlabel("Frame")
plt.ylabel("Speed (km/h)")
plt.title(f"Vehicle speed - PandaSet sequence {SEQUENCE}")
plt.grid(True, alpha=0.3)
plt.savefig(os.path.join(FIG_DIR, "fig4_speed.png"), dpi=300, bbox_inches="tight")  # article figure
plt.show()

## 5. Ground filtering and bird's-eye view (BEV) of the road

Selects points near the ground plane in front of the vehicle, splits them into left and right road boundaries, estimates the road width and the lateral offset of the vehicle from the road center, and plots a bird's-eye view. See Section 2 regarding the coordinate frame.

In [ ]:
# Ground-point filtering, road-boundary extraction and bird's-eye-view plot for one LiDAR frame.

def extract_road_boundaries(cloud):
    """
    Select ground points ahead of the vehicle and split them into left and right boundaries.

    Returns the left points, the right points and the estimated ground height.
    """
    # Ground height estimated as the 10th percentile of Z; keep points within [-0.05, +0.4] m of it
    z_base = np.percentile(cloud[:, 2], 10)
    ground_mask = (cloud[:, 2] > z_base - 0.05) & (cloud[:, 2] < z_base + 0.4)
    ground = cloud[ground_mask]

    # Region of interest: 2-35 m ahead and within +/- 6 m laterally
    roi_mask = (ground[:, 0] > 2) & (ground[:, 0] < 35) & (np.abs(ground[:, 1]) < 6)
    road = ground[roi_mask]

    # Left (y > 0) and right (y < 0) halves of the road
    left = road[road[:, 1] > 0]
    right = road[road[:, 1] < 0]

    def remove_outliers(p):
        # Remove points whose lateral position deviates more than 1.5 std from the mean
        return p[np.abs(p[:, 1] - np.mean(p[:, 1])) < 1.5 * np.std(p[:, 1])] if len(p) > 10 else p

    return remove_outliers(left), remove_outliers(right), z_base


def estimate_center_offset(left, right):
    """
    Estimate the lateral offset (m) of the road center relative to the vehicle,
    averaging the midpoint between left and right boundaries in 1 m slices from 5 to 30 m ahead.
    """
    centers = []
    for x in np.arange(5, 30, 1):
        p_left = left[(left[:, 0] >= x) & (left[:, 0] < x + 1)]
        p_right = right[(right[:, 0] >= x) & (right[:, 0] < x + 1)]
        if len(p_left) > 0 and len(p_right) > 0:
            centers.append((np.mean(p_left[:, 1]) + np.mean(p_right[:, 1])) / 2)
    return np.mean(centers) if centers else 0.0


def plot_bev(left, right, center_offset):
    """Bird's-eye view of the road boundaries, the estimated centerline and the ego vehicle."""
    plt.figure(figsize=(12, 8))

    # Boundary points (small marker size to avoid clutter)
    plt.scatter(left[:, 0], left[:, 1], s=1, c="#3498db", label="Left boundary (LiDAR points)", alpha=0.5)
    plt.scatter(right[:, 0], right[:, 1], s=1, c="#e74c3c", label="Right boundary (LiDAR points)", alpha=0.5)

    # Estimated road centerline (constant lateral offset)
    x_axis = np.linspace(5, 35, 10)
    plt.plot(x_axis, np.full_like(x_axis, center_offset), "--", c="gold", linewidth=2,
             label=f"Road centerline (offset: {center_offset:.2f} m)")

    # Ego vehicle at the origin, pointing forward (+X)
    plt.scatter(0, 0, marker="s", s=200, c="black", label="Ego vehicle")
    plt.arrow(0, 0, 2, 0, head_width=0.5, head_length=0.5, fc="black", ec="black")

    plt.title("Bird's-eye view (BEV) - PandaSet road segmentation", fontsize=14)
    plt.xlabel("Forward distance (m)")
    plt.ylabel("Lateral distance (m)")
    plt.grid(True, linestyle=":", alpha=0.6)
    plt.legend(loc="upper right")
    plt.axis("equal")  # keep the true aspect ratio of the road
    plt.xlim(-5, 40)
    plt.ylim(-10, 10)
    plt.savefig(os.path.join(FIG_DIR, "fig3_bev.png"), dpi=300, bbox_inches="tight")  # article figure
    plt.show()


# --- Run on one frame ---
frame_path = os.path.join(PANDASET_ROOT, SEQUENCE, "lidar", "00.pkl")
points = pd.read_pickle(frame_path)[["x", "y", "z"]].values
# Transform from world coordinates to the vehicle-aligned frame (x forward, y left)
points = mundo_a_vehiculo(points, (sensor_x[0], sensor_y[0]), sensor_yaw[0])

left_pts, right_pts, ground_z = extract_road_boundaries(points)
offset = estimate_center_offset(left_pts, right_pts)

print("--- Road analysis ---")
print(f"Ground height: {ground_z:.2f} m")
print(f"Road width:    {np.mean(left_pts[:, 1]) - np.mean(right_pts[:, 1]):.2f} m")
print(f"Center offset: {offset:.2f} m")
# A positive offset means the road center lies to the left (+Y), i.e., the vehicle is right of center
print("Vehicle position: right of the road center." if offset > 0.5 else
      "Vehicle position: centered or left of the road center.")

plot_bev(left_pts, right_pts, offset)

In [ ]:
# Package the article figures (fig2_point_cloud, fig3_bev, fig4_speed) into a zip and download it.
import shutil
from google.colab import files

shutil.make_archive("figures_exploration", "zip", FIG_DIR)
files.download("figures_exploration.zip")